In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
# pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기: 2-5 train_val_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

print(train.shape, val.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(616293, 35) (133316, 35)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np

from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.metrics import root_mean_squared_error

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [5]:
# ============================================================
# 3. Tidal / Normal 데이터 분리
# ============================================================

# 조석 영향 의심 관측소 9개 (기존과 동일)
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# Train 분리
tidal_train = train.loc[
    train['station_id'].isin(tidal_stations)
].copy()

normal_train = train.loc[
    ~train['station_id'].isin(tidal_stations)
].copy()

# Val 분리
tidal_val = val.loc[
    val['station_id'].isin(tidal_stations)
].copy()

normal_val = val.loc[
    ~val['station_id'].isin(tidal_stations)
].copy()

# 데이터 개수 확인
print('[Train]')
print(f'분리 전 총계 : {len(train):,}')
print(f'분리 후 Normal: {len(normal_train):,}')
print(f'분리 후 Tidal : {len(tidal_train):,}')

print('\n[Validation]')
print(f'분리 전 총계 : {len(val):,}')
print(f'분리 후 Normal: {len(normal_val):,}')
print(f'분리 후 Tidal : {len(tidal_val):,}')

[Train]
분리 전 총계 : 616,293
분리 후 Normal: 604,608
분리 후 Tidal : 11,685

[Validation]
분리 전 총계 : 133,316
분리 후 Normal: 130,750
분리 후 Tidal : 2,566


In [6]:
# ============================================================
# 4. 독립변수 / 종속변수 분리
# ============================================================

# Tidal
X_tidal_train = tidal_train[num_cols_fe + ['station_id']].copy()
X_tidal_val = tidal_val[num_cols_fe + ['station_id']].copy()

y_tidal_train = tidal_train[target_col]
y_tidal_val = tidal_val[target_col]

# Normal
X_normal_train = normal_train[num_cols_fe + ['station_id']].copy()
X_normal_val = normal_val[num_cols_fe + ['station_id']].copy()

y_normal_train = normal_train[target_col]
y_normal_val = normal_val[target_col]

# CatBoost 범주형 변수 처리
for X in [
    X_tidal_train, X_tidal_val,
    X_normal_train, X_normal_val
]:
    X['station_id'] = X['station_id'].astype(str)

In [9]:
# ============================================================
# 5. Normal / Tidal CatBoost 학습 및 Validation 예측
# ============================================================

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
import pandas as pd

preds_val = {}
results_val = {}

# 1. Normal: 기존 확정 CatBoost 모델
model_normal = CatBoostRegressor(
    iterations=3000,
    random_seed=42,
    cat_features=['station_id'],
    verbose=200
)

model_normal.fit(
    X_normal_train, y_normal_train,
    eval_set=(X_normal_val, y_normal_val),
    early_stopping_rounds=100
)

# 2. Tidal: 기본형 CatBoost
model_tidal = CatBoostRegressor(
    random_seed=42,
    cat_features=['station_id'],
    verbose=200
)

model_tidal.fit(X_tidal_train, y_tidal_train)

# 3. 그룹별 예측 및 성능 저장
for group, model in [
    ('normal', model_normal),
    ('tidal', model_tidal)
]:
    X_train = X_normal_train if group == 'normal' else X_tidal_train
    X_val = X_normal_val if group == 'normal' else X_tidal_val
    y_train = y_normal_train if group == 'normal' else y_tidal_train
    y_val = y_normal_val if group == 'normal' else y_tidal_val
    pred_val = model.predict(X_val)
    pred_train = model.predict(X_train)

    preds_val[f'{group}_CatBoost'] = pd.Series(
        pred_val, index=X_val.index
    )

    results_val[f'{group}_CatBoost'] = {
        'Val RMSE': root_mean_squared_error(y_val, pred_val),
        'Train RMSE': root_mean_squared_error(y_train, pred_train)
    }

display(pd.DataFrame.from_dict(results_val, orient='index'))

Learning rate set to 0.07121
0:	learn: 0.2846285	test: 0.2243682	best: 0.2243682 (0)	total: 369ms	remaining: 18m 27s
200:	learn: 0.2179375	test: 0.1896694	best: 0.1896694 (200)	total: 36.5s	remaining: 8m 27s
400:	learn: 0.2078211	test: 0.1870848	best: 0.1870649 (399)	total: 1m 11s	remaining: 7m 42s
600:	learn: 0.2017569	test: 0.1867552	best: 0.1867185 (590)	total: 1m 51s	remaining: 7m 26s
800:	learn: 0.1969781	test: 0.1866371	best: 0.1865962 (742)	total: 2m 29s	remaining: 6m 49s
1000:	learn: 0.1929121	test: 0.1864860	best: 0.1864503 (952)	total: 3m 3s	remaining: 6m 5s
1200:	learn: 0.1895492	test: 0.1862929	best: 0.1862671 (1196)	total: 3m 43s	remaining: 5m 34s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.186251702
bestIteration = 1211

Shrink model to first 1212 iterations.
Learning rate set to 0.060376
0:	learn: 1.6844914	total: 52.2ms	remaining: 52.1s
200:	learn: 0.3696026	total: 11.1s	remaining: 43.9s
400:	learn: 0.3075402	total: 19.3s	remaining: 28.8s
600:	l

,Val RMSE,Train RMSE
normal_CatBoost,0.186252,0.188678
tidal_CatBoost,0.346101,0.223757


In [11]:
# ============================================================
# 5-1. Normal + Tidal CatBoost 통합 Validation 성능
# ============================================================

# 1. 그룹별 예측값 통합 (기존 행 인덱스 유지)
pred_val_total = pd.concat([
    preds_val['normal_CatBoost'],
    preds_val['tidal_CatBoost']
]).reindex(val.index)

# 2. 전체 Validation RMSE 계산
rmse_val_total = root_mean_squared_error(
    val[target_col],
    pred_val_total
)

# 3. 결과 출력
print('[Normal + Tidal CatBoost]')
print(f'Normal Val RMSE : {results_val["normal_CatBoost"]["Val RMSE"]:.6f}')
print(f'Tidal Val RMSE  : {results_val["tidal_CatBoost"]["Val RMSE"]:.6f}')
print(f'전체 Val RMSE   : {rmse_val_total:.6f}')

[Normal + Tidal CatBoost]
Normal Val RMSE : 0.186252
Tidal Val RMSE  : 0.346101
전체 Val RMSE   : 0.190598


In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor, log_evaluation
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import gc

# ============================================================
# 6. Tidal 데이터 부스팅모델별 성능 비교
# ============================================================

preds_tidal = {}
results_tidal = {}

for name in ['CatBoost', 'LightGBM', 'XGBoost']:

    print(f'\n========== {name} 학습 ==========')

    # 1. 모델 정의 및 범주형 변수 처리
    if name == 'CatBoost':
        model = CatBoostRegressor(
            random_seed=42,
            cat_features=['station_id'],
            verbose=200
        )

        X_train = X_tidal_train
        X_val = X_tidal_val

    else:
        X_train = X_tidal_train.copy()
        X_val = X_tidal_val.copy()

        # station_id 범주형 자료형 변환
        X_train['station_id'] = X_train['station_id'].astype('category')

        # Train의 범주 목록을 Val에도 동일하게 적용
        X_val['station_id'] = pd.Categorical(
            X_val['station_id'],
            categories=X_train['station_id'].cat.categories
        )

        if name == 'LightGBM':
            model = LGBMRegressor(
                random_state=42,
                verbosity=1
            )

        elif name == 'XGBoost':
            model = XGBRegressor(
                random_state=42,
                enable_categorical=True,
                eval_metric='rmse'
            )

    # 2. 모델별 학습
    if name == 'LightGBM':
        model.fit(
            X_train, y_tidal_train,
            eval_set=[(X_train, y_tidal_train)],
            eval_metric='rmse',
            callbacks=[log_evaluation(10)]
        )

    elif name == 'XGBoost':
        model.fit(
            X_train, y_tidal_train,
            eval_set=[(X_train, y_tidal_train)],
            verbose=10
        )

    else:
        model.fit(X_train, y_tidal_train)

    # 3. 예측
    pred_val = model.predict(X_val)
    pred_train = model.predict(X_train)

    # 4. RMSE 계산
    rmse_val = root_mean_squared_error(y_tidal_val, pred_val)
    rmse_train = root_mean_squared_error(y_tidal_train, pred_train)

    # 5. 결과 저장
    preds_tidal[name] = pd.Series(
        pred_val, index=X_tidal_val.index
    )

    results_tidal[name] = {
        'Val RMSE': rmse_val,
        'Train RMSE': rmse_train,
        'Gap': rmse_val - rmse_train
    }

    print(f'\n{name} 결과')
    print('Val RMSE:', rmse_val)
    print('Train RMSE:', rmse_train)
    print('Gap:', rmse_val - rmse_train)

    # 6. 모델 메모리 정리
    del model
    gc.collect()

# 7. 모델별 성능 비교표
result_df = pd.DataFrame.from_dict(
    results_tidal,
    orient='index'
)

display(result_df.sort_values('Val RMSE'))


========== CatBoost 학습 ==========
Learning rate set to 0.060376
0:	learn: 1.6844914	total: 44.6ms	remaining: 44.6s
200:	learn: 0.3696026	total: 9.08s	remaining: 36.1s
400:	learn: 0.3075402	total: 19.2s	remaining: 28.6s
600:	learn: 0.2705500	total: 29.4s	remaining: 19.5s
800:	learn: 0.2442506	total: 38.6s	remaining: 9.6s
999:	learn: 0.2234061	total: 47.1s	remaining: 0us

CatBoost 결과
Val RMSE: 0.3461011209956592
Train RMSE: 0.2237569706010531
Gap: 0.12234415039460611

========== LightGBM 학습 ==========
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001648 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5794
[LightGBM] [Info] Number of data points in the train set: 11685, number of used features: 32
[LightGBM] [Info] Start training from score 1.142274
[10]	training's rmse: 0.762176	training's l2: 0.580912
[20]	training's rmse: 0.45983	training's l2: 0.211444
[30]	training's rmse: 0.373277	training's l

c:\Users\20162180\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[40]	training's rmse: 0.339845	training's l2: 0.115495
[50]	training's rmse: 0.319766	training's l2: 0.10225
[60]	training's rmse: 0.304588	training's l2: 0.0927739
[70]	training's rmse: 0.292395	training's l2: 0.0854947
[80]	training's rmse: 0.282844	training's l2: 0.0800009
[90]	training's rmse: 0.272682	training's l2: 0.0743554
[100]	training's rmse: 0.264483	training's l2: 0.069951

LightGBM 결과
Val RMSE: 0.36324832454567796
Train RMSE: 0.2644825654792503
Gap: 0.09876575906642765

========== XGBoost 학습 ==========
[0]	validation_0-rmse:1.29553
[10]	validation_0-rmse:0.33650
[20]	validation_0-rmse:0.28493
[30]	validation_0-rmse:0.25735
[40]	validation_0-rmse:0.23561
[50]	validation_0-rmse:0.21598
[60]	validation_0-rmse:0.19830
[70]	validation_0-rmse:0.18314
[80]	validation_0-rmse:0.17252
[90]	validation_0-rmse:0.16412
[99]	validation_0-rmse:0.15459

XGBoost 결과
Val RMSE: 0.375795864669198
Train RMSE: 0.15458867769737183
Gap: 0.22120718697182618


,Val RMSE,Train RMSE,Gap
CatBoost,0.346101,0.223757,0.122344
LightGBM,0.363248,0.264483,0.098766
XGBoost,0.375796,0.154589,0.221207


In [13]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import gc

# ============================================================
# 6-2. Tidal 데이터 배깅모델별 성능 비교
# ============================================================

# 1. OneHotEncoder 정의 및 학습
ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

ohe.fit(X_tidal_train[['station_id']])

# 2. Train / Val 원핫인코딩
X_tidal_train_ohe = pd.concat([
    X_tidal_train[num_cols_fe],
    pd.DataFrame(
        ohe.transform(X_tidal_train[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_train.index
    )
], axis=1)

X_tidal_val_ohe = pd.concat([
    X_tidal_val[num_cols_fe],
    pd.DataFrame(
        ohe.transform(X_tidal_val[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_val.index
    )
], axis=1)

# 3. RF / ExtraTrees 기본모델 비교
# 앞에서 학습한 CatBoost, LightGBM, XGBoost 결과에 추가
# preds_tidal, results_tidal 딕셔너리 유지

for name in ['RandomForest', 'ExtraTrees']:

    print(f'\n========== {name} 학습 ==========')

    # 4. 모델 정의
    if name == 'RandomForest':
        model = RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            verbose=2
        )

    elif name == 'ExtraTrees':
        model = ExtraTreesRegressor(
            random_state=42,
            n_jobs=-1,
            verbose=2
        )

    # 5. 모델 학습
    model.fit(X_tidal_train_ohe, y_tidal_train)

    # 6. 예측
    pred_val = model.predict(X_tidal_val_ohe)
    pred_train = model.predict(X_tidal_train_ohe)

    # 7. RMSE 계산
    rmse_val = root_mean_squared_error(y_tidal_val, pred_val)
    rmse_train = root_mean_squared_error(y_tidal_train, pred_train)

    # 8. 결과 저장
    preds_tidal[name] = pd.Series(
        pred_val, index=X_tidal_val.index
    )

    results_tidal[name] = {
        'Val RMSE': rmse_val,
        'Train RMSE': rmse_train,
        'Gap': rmse_val - rmse_train
    }

    print(f'\n{name} 결과')
    print('Val RMSE:', rmse_val)
    print('Train RMSE:', rmse_train)
    print('Gap:', rmse_val - rmse_train)

    # 9. 메모리 정리
    del model
    gc.collect()

# 10. 전체 5개 모델 성능 비교
result_df = pd.DataFrame.from_dict(
    results_tidal,
    orient='index'
)

display(result_df.sort_values('Val RMSE'))


========== RandomForest 학습 ==========
building tree 4 of 100
building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 6 of 100
building tree 5 of 100


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 6 concurrent workers.


building tree 7 of 100building tree 8 of 100

building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100


[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:    1.2s


building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100building tree 80 

...

[Parallel(n_jobs=-1)]: Done  29 tasks      | elapsed:    0.6s


building tree 39 of 100
building tree 40 of 100
building tree 41 of 100
building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100
building tree 80

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    1.9s finished
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=6)]: Using backend ThreadingBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done  29 tasks      | elapsed:    0.0s
[Parallel(n_jobs=6)]: Done 100 out of 100 | elapsed:    0.0s finished



ExtraTrees 결과
Val RMSE: 0.34202678604656644
Train RMSE: 2.9337950090255696e-15
Gap: 0.3420267860465635


,Val RMSE,Train RMSE,Gap
ExtraTrees,0.342027,2.933795e-15,0.342027
CatBoost,0.346101,2.237570e-01,0.122344
LightGBM,0.363248,2.644826e-01,0.098766
RandomForest,0.364819,1.299629e-01,0.234856
XGBoost,0.375796,1.545887e-01,0.221207


In [14]:
# ============================================================
# 7. Tidal ExtraTrees, Normal은 Cat학습 및 통합 Validation 성능 비교
# ============================================================

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.preprocessing import OneHotEncoder

# 1. OneHotEncoder
ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

ohe.fit(X_tidal_train[['station_id']])

X_tidal_train_ohe = pd.concat([
    X_tidal_train[num_cols_fe],
    pd.DataFrame(
        ohe.transform(X_tidal_train[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_train.index
    )
], axis=1)

X_tidal_val_ohe = pd.concat([
    X_tidal_val[num_cols_fe],
    pd.DataFrame(
        ohe.transform(X_tidal_val[['station_id']]),
        columns=ohe.get_feature_names_out(),
        index=X_tidal_val.index
    )
], axis=1)

# 2. ExtraTrees 기본모델 학습
model_et = ExtraTreesRegressor(
    random_state=42,
    n_jobs=-1
)

model_et.fit(X_tidal_train_ohe, y_tidal_train)

# 3. 예측
pred_val = model_et.predict(X_tidal_val_ohe)
pred_train = model_et.predict(X_tidal_train_ohe)

# 4. 결과 저장
preds_val['tidal_ExtraTrees'] = pd.Series(
    pred_val, index=X_tidal_val.index
)

results_val['tidal_ExtraTrees'] = {
    'Val RMSE': root_mean_squared_error(y_tidal_val, pred_val),
    'Train RMSE': root_mean_squared_error(y_tidal_train, pred_train)
}

# 5. Tidal 모델별 성능 비교
print('[Tidal 모델 성능]')
display(pd.DataFrame.from_dict(results_val, orient='index'))

# 6. 전체 Validation 예측값 통합 및 RMSE 비교
pipeline_results = {}

for name, tidal_key in [
    ('Normal CAT + Tidal CAT', 'tidal_CatBoost'),
    ('Normal CAT + Tidal ET', 'tidal_ExtraTrees')
]:

    pred_val_total = pd.concat([
        preds_val['normal_CatBoost'],
        preds_val[tidal_key]
    ]).reindex(val.index)

    pipeline_results[name] = {
        'Val RMSE': root_mean_squared_error(
            val[target_col],
            pred_val_total
        )
    }

# 7. 통합 Validation 성능 비교표
print('\n[전체 Validation 파이프라인 성능 비교]')

pipeline_df = pd.DataFrame.from_dict(
    pipeline_results,
    orient='index'
)

display(pipeline_df.sort_values('Val RMSE'))

[Tidal 모델 성능]


,Val RMSE,Train RMSE
normal_CatBoost,0.186252,1.886777e-01
tidal_CatBoost,0.346101,2.237570e-01
tidal_ExtraTrees,0.342027,2.933798e-15



[전체 Validation 파이프라인 성능 비교]


,Val RMSE
Normal CAT + Tidal ET,0.190456
Normal CAT + Tidal CAT,0.190598


In [15]:
# ============================================================
# 8. Tidal CAT + ET 앙상블 가중치별 전체 Validation RMSE 비교
# ============================================================

ensemble_results = []

for w_cat in np.arange(0, 1.001, 0.05):

    # 1. Tidal 앙상블 예측
    pred_tidal_ensemble = (
        preds_val['tidal_CatBoost'] * w_cat
        + preds_val['tidal_ExtraTrees'] * (1 - w_cat)
    )

    # 2. Normal CAT + Tidal 앙상블 통합
    pred_val_total = pd.concat([
        preds_val['normal_CatBoost'],
        pred_tidal_ensemble
    ]).reindex(val.index)

    # 3. 전체 Validation RMSE 계산
    rmse_val = root_mean_squared_error(
        val[target_col],
        pred_val_total
    )

    # 4. 결과 저장
    ensemble_results.append({
        'CAT 비율': round(w_cat, 2),
        'ET 비율': round(1 - w_cat, 2),
        'Val RMSE': rmse_val
    })

# 5. 가중치별 성능 비교
ensemble_df = pd.DataFrame(ensemble_results)

display(
    ensemble_df.sort_values('Val RMSE')
)

,CAT 비율,ET 비율,Val RMSE
9,0.45,0.55,0.190178
8,0.40,0.60,0.190181
10,0.50,0.50,0.190182
7,0.35,0.65,0.190192
11,0.55,0.45,0.190192
6,0.30,0.70,0.190209
12,0.60,0.40,0.190210
5,0.25,0.75,0.190233
13,0.65,0.35,0.190234
4,0.20,0.80,0.190264


In [8]:

# ============================================================
# 9. Normal / Tidal CatBoost 5-Fold 교차검증 및 Validation 앙상블
# ============================================================

# 1. 고정 Fold 매핑 파일 불러오기
# 기존 교차검증에서 확정된 강우사상별 Fold 번호를 그대로 사용
# GroupKFold를 다시 실행하여 Fold를 재배정하지 않음

kf_fold_fixed = pd.read_csv(
    PROJECT_ROOT / 'config' / '0_kf_event_fold_match.csv'
)

# 고정 Fold 매핑 검증
assert not kf_fold_fixed['event_id'].duplicated().any(), 'event_id 중복 발생'
assert set(kf_fold_fixed['fold_num']) == {1, 2, 3, 4, 5}, 'Fold 번호 확인 필요'

# Train의 event_id를 기준으로 기존 Fold 번호 연결
# Normal / Tidal 모두 동일한 강우사상 Fold 배정을 사용
kf_fold = train['event_id'].map(
    kf_fold_fixed.set_index('event_id')['fold_num']
)

# Fold 매핑 누락 확인
assert kf_fold.notna().all(), 'Fold 매핑 누락 발생'

# 예측값 저장 시 인덱스 정합성 검증
assert train.index.is_unique, 'Train 인덱스 중복 발생'
assert val.index.is_unique, 'Validation 인덱스 중복 발생'

print('[Fold별 데이터 수]')
print(kf_fold.value_counts().sort_index())


# ============================================================
# 2. 교차검증 결과 및 예측값 저장 공간 생성
# ============================================================

# Fold별 Train / OOF RMSE, 최적 iteration 저장
kf_results = []

# 전체 Train OOF 예측값
# 각 행은 자신이 검증 데이터로 배정된 Fold 모델의 예측값을 가짐
kf_oof_pred = pd.Series(np.nan, index=train.index)

# Validation 예측값
# 각 Fold 모델이 독립 Validation 전체를 예측한 결과 저장
# 최종적으로 Fold 1~5 예측값을 평균하여 앙상블
kf_val_pred = pd.DataFrame(
    np.nan,
    index=val.index,
    columns=[f'fold_{i}' for i in range(1, 6)]
)


# ============================================================
# 3. 고정 Fold 기반 5-Fold 교차검증
# ============================================================

for fold_num in range(1, 6):

    print(f'\n========== Fold {fold_num} ==========')

    # 동일한 Fold에서 Normal / Tidal 각각 모델링
    for group, X_train, X_val, y_train, y_val in [
        (
            'normal',
            X_normal_train, X_normal_val,
            y_normal_train, y_normal_val
        ),
        (
            'tidal',
            X_tidal_train, X_tidal_val,
            y_tidal_train, y_tidal_val
        )
    ]:

        # ----------------------------------------------------
        # 3-1. Fold별 학습 / 검증 데이터 분할
        # ----------------------------------------------------

        # 해당 Fold 번호에 속하는 강우사상은 검증 데이터로 사용
        # 나머지 4개 Fold는 학습 데이터로 사용
        train_idx = kf_fold.loc[X_train.index] != fold_num  # train_idx는 불리언 조건문
        val_idx = kf_fold.loc[X_train.index] == fold_num

        X_kf_tr = X_train.loc[train_idx]
        X_kf_va = X_train.loc[val_idx]

        y_kf_tr = y_train.loc[train_idx]
        y_kf_va = y_train.loc[val_idx]

        # ----------------------------------------------------
        # 3-2. 모델 정의
        # ----------------------------------------------------

        # Normal / Tidal 동일한 CatBoost 학습 조건 적용
        model_cat_kf = CatBoostRegressor(
            random_seed=42,
            iterations=3000,
            cat_features=['station_id']
        )

        # ----------------------------------------------------
        # 3-3. 모델 학습
        # ----------------------------------------------------

        # Fold의 검증 데이터는 조기 종료 기준으로 사용
        # 독립 Validation 데이터는 학습에 사용하지 않음
        model_cat_kf.fit(
            X_kf_tr,
            y_kf_tr,
            early_stopping_rounds=100,
            eval_set=(X_kf_va, y_kf_va),
            verbose=200
        )

        # ----------------------------------------------------
        # 3-4. Fold별 예측
        # ----------------------------------------------------

        # 해당 Fold의 검증 데이터 예측 (OOF)
        pred_kf_va = model_cat_kf.predict(X_kf_va)

        # 해당 Fold의 학습 데이터 예측
        pred_kf_tr = model_cat_kf.predict(X_kf_tr)

        # 독립 Validation 전체 예측
        # Fold마다 별도로 예측하여 총 5개 결과 생성
        pred_val = model_cat_kf.predict(X_val)

        # ----------------------------------------------------
        # 3-5. 예측값 저장
        # ----------------------------------------------------

        # OOF 예측값을 원본 Train 인덱스 위치에 저장
        kf_oof_pred.loc[X_kf_va.index] = pred_kf_va

        # 독립 Validation 예측값을 Fold별 컬럼에 저장
        kf_val_pred.loc[
            X_val.index, f'fold_{fold_num}'
        ] = pred_val

        # ----------------------------------------------------
        # 3-6. Fold별 RMSE 평가
        # ----------------------------------------------------

        val_rmse = root_mean_squared_error(
            y_kf_va, pred_kf_va
        )

        train_rmse = root_mean_squared_error(
            y_kf_tr, pred_kf_tr
        )

        # 평가 결과 저장
        kf_results.append({
            'fold': fold_num,
            'group': group,
            'val_rmse': val_rmse,
            'train_rmse': train_rmse,
            'gap': val_rmse - train_rmse,
            'best_iteration': model_cat_kf.best_iteration_
        })

        # 결과 출력
        print(
            f'{group} | Fold {fold_num} | '
            f'Val={val_rmse:.4f} | '
            f'Train={train_rmse:.4f} | '
            f'Gap={val_rmse - train_rmse:.4f} | '
            f'Best iter={model_cat_kf.best_iteration_}'
        )


# ============================================================
# 4. 전체 OOF 및 Validation 앙상블 평가
# ============================================================

# 4-1. 예측값 누락 검증
print('\nOOF 예측 누락:', kf_oof_pred.isna().sum())
print('Validation 예측 누락:', kf_val_pred.isna().sum().sum())

assert kf_oof_pred.notna().all(), 'OOF 예측 누락 발생'
assert kf_val_pred.notna().all().all(), 'Validation 예측 누락 발생'


# 4-2. 전체 Train OOF RMSE
# Normal / Tidal OOF 예측값을 합쳐 전체 Train 기준 평가
oof_rmse = root_mean_squared_error(
    train[target_col],
    kf_oof_pred
)


# 4-3. Validation 5개 Fold 모델 앙상블
# 각 Validation 행에 대한 5개 예측값의 산술평균
pred_val_ensemble = kf_val_pred.mean(axis=1)


# 4-4. 전체 Validation RMSE
val_rmse_total = root_mean_squared_error(
    val[target_col],
    pred_val_ensemble
)


# ============================================================
# 5. 결과 확인
# ============================================================

# 5-1. Fold별 교차검증 성능 비교
kf_result_df = pd.DataFrame(kf_results)

print('\n[Fold별 교차검증 결과]')
display(kf_result_df)


# 5-2. Normal / Tidal별 OOF 및 Validation RMSE
print('\n[그룹별 성능]')

for group, X_train, X_val, y_train, y_val in [
    (
        'normal',
        X_normal_train, X_normal_val,
        y_normal_train, y_normal_val
    ),
    (
        'tidal',
        X_tidal_train, X_tidal_val,
        y_tidal_train, y_tidal_val
    )
]:
    group_oof_rmse = root_mean_squared_error(
        y_train,
        kf_oof_pred.loc[X_train.index]
    )

    group_val_rmse = root_mean_squared_error(
        y_val,
        pred_val_ensemble.loc[X_val.index]
    )

    print(
        f'{group} | '
        f'OOF RMSE={group_oof_rmse:.6f} | '
        f'Validation RMSE={group_val_rmse:.6f}'
    )


# 5-3. 전체 성능 출력
print('\n[전체 최종 성능]')
print(f'전체 OOF RMSE        : {oof_rmse:.6f}')
print(f'전체 Validation RMSE : {val_rmse_total:.6f}')


[Fold별 데이터 수]
event_id
1    122996
2    123399
3    123399
4    123216
5    123283
Name: count, dtype: int64

========== Fold 1 ==========
Learning rate set to 0.068766
0:	learn: 0.2752341	test: 0.3199435	best: 0.3199435 (0)	total: 215ms	remaining: 10m 43s
200:	learn: 0.2055879	test: 0.2694372	best: 0.2694372 (200)	total: 36.1s	remaining: 8m 22s
400:	learn: 0.1950586	test: 0.2660327	best: 0.2660327 (400)	total: 1m 6s	remaining: 7m 13s
600:	learn: 0.1885954	test: 0.2653412	best: 0.2653038 (595)	total: 1m 36s	remaining: 6m 24s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2649676576
bestIteration = 690

Shrink model to first 691 iterations.
normal | Fold 1 | Val=0.2650 | Train=0.1852 | Gap=0.0797 | Best iter=690
Learning rate set to 0.03698
0:	learn: 1.6741950	test: 1.8746893	best: 1.8746893 (0)	total: 37.5ms	remaining: 1m 52s
200:	learn: 0.3966562	test: 0.5275649	best: 0.5275649 (200)	total: 8.5s	remaining: 1m 58s
400:	learn: 0.3374297	test: 0.4975205	best: 0.4975

,fold,group,val_rmse,train_rmse,gap,best_iteration
0,1,normal,0.264968,0.185222,0.079746,690
1,1,tidal,0.475109,0.242733,0.232376,1169
2,2,normal,0.138159,0.206136,-0.067977,896
3,2,tidal,0.359429,0.211879,0.147550,1651
4,3,normal,0.222469,0.174849,0.047620,2153
5,3,tidal,0.344497,0.154577,0.189920,2975
6,4,normal,0.268136,0.187453,0.080682,566
7,4,tidal,0.372886,0.157071,0.215815,2918
8,5,normal,0.214383,0.189199,0.025184,1118
9,5,tidal,0.363554,0.159285,0.204270,2819



[그룹별 성능]
normal | OOF RMSE=0.226512 | Validation RMSE=0.185111
tidal | OOF RMSE=0.386370 | Validation RMSE=0.339517

[전체 최종 성능]
전체 OOF RMSE        : 0.230576
전체 Validation RMSE : 0.189276
